In [1]:
import pandas as pd
import numpy as np
import gc

In [2]:
df_Trans = pd.read_parquet('transactions_capping.parquet')
df_TransItem = pd.read_parquet('transaction_items_cleaned.parquet')
df_Users = pd.read_parquet('users_cleaned.parquet')
df_menu = pd.read_parquet('menu_cleaned.parquet')
df_stores = pd.read_parquet('stores_cleaned.parquet')

# Data Master

In [3]:
df_MasterTrans = (
    df_Trans
    .merge(df_Users, on='user_id', how='left')
    .merge(df_stores, on='store_id', how='left')
)

In [4]:
df_Master = (
    df_TransItem
    .merge(df_menu, on='item_id', how='left')
    .merge(
        df_MasterTrans,
        on='transaction_id',
        how='left'
    )
)

In [5]:
len(df_MasterTrans)

14623691

In [6]:
len(df_Master)

26885688

## Validating eror

In [7]:
def validate_and_clean_master(df_master, df_header, df_item_raw):
    print("=== MASTER VALIDATION ===\n")

    # 1. Row Count Integrity
    row_diff = len(df_master) - len(df_item_raw)

    if row_diff == 0:
        print(f"Row Count Check: OK ({len(df_master):,} rows)")
    else:
        print(f"Row Count Check: Mismatch ({row_diff:,} row difference)")

    # 2. Orphan Check
    print("\nOrphan Check:")

    orphans = {
        'Menu': df_master['item_name'].isnull().sum(),
        'Store': df_master['store_name'].isnull().sum(),
        'Header': df_master['original_amount'].isnull().sum()
    }

    for key, val in orphans.items():
        print(f"- {key}: {val:,} orphan records")

    # 3. Financial Audit
    print("\nFinancial Audit:")

    audit_items = (
        df_master
        .groupby('transaction_id')['subtotal']
        .sum()
        .reset_index()
    )

    audit_merge = audit_items.merge(
        df_header[['transaction_id', 'original_amount']],
        on='transaction_id',
        how='inner'
    )

    audit_merge['diff'] = (
        audit_merge['original_amount'] -
        audit_merge['subtotal']
    ).abs()

    mismatches = audit_merge[audit_merge['diff'] > 0.01]

    if len(mismatches) == 0:
        print(f"Balanced Transactions: {len(audit_merge):,}")
    else:
        print(f"Mismatched Transactions: {len(mismatches):,}")

    # 4. Remove Redundant Columns
    print("\nColumn Cleanup:")

    redundant_cols = [
        col for col in df_master.columns
        if col.endswith('_x') or col.endswith('_y')
    ]

    if 'created_at_x' in df_master.columns:
        df_master['created_at'] = df_master['created_at_x']

    if redundant_cols:
        df_master.drop(
            columns=redundant_cols,
            inplace=True,
            errors='ignore'
        )

        print(f"Removed Columns: {', '.join(redundant_cols)}")
    else:
        print("No redundant columns found")

    print("\n=== VALIDATION COMPLETED ===")

    return df_master


# Execute
df_Master = validate_and_clean_master(
    df_Master,
    df_MasterTrans,
    df_TransItem
)

=== MASTER VALIDATION ===

Row Count Check: OK (26,885,688 rows)

Orphan Check:
- Menu: 0 orphan records
- Store: 0 orphan records
- Header: 0 orphan records

Financial Audit:
Balanced Transactions: 14,623,691

Column Cleanup:
Removed Columns: created_at_x, created_at_y

=== VALIDATION COMPLETED ===


In [10]:
df_payment = pd.read_parquet('paymentsUnique.parquet')

df_Master = df_Master.merge(
    df_payment,
    left_on='payment_method_id',
    right_on='method_id',
    how='left'
)

In [11]:
df_Master.to_parquet('df_Master_Final.parquet', index=False)

In [12]:
del df_Trans, df_TransItem, df_Users, df_menu, df_stores, df_MasterTrans
gc.collect()

NameError: name 'df_Trans' is not defined